In [ ]:
# Part 3 — GitHub Actions + Google Colab compatible
# The video editing / FFmpeg composition logic below is preserved from the supplied notebook.
# Only runtime setup, input/output paths, and failure reporting are adapted.

import os
import sys
import shutil
import subprocess
from pathlib import Path
import importlib.util

IS_GITHUB_ACTIONS = os.environ.get("GITHUB_ACTIONS", "").strip().lower() == "true"
WORKSPACE = Path(os.environ.get("GITHUB_WORKSPACE") or Path.cwd()).resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)

ASSETS_DIR = WORKSPACE / "assets"
INPUT_DIR = WORKSPACE / "clips"
OUTPUT_DIR = WORKSPACE / "Ready_!"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Install missing Python packages only when necessary. In GitHub Actions the workflow
# installs these before executing this notebook; Colab gets the same convenience as before.
packages = {
    "gdown": "gdown",
    "tqdm": "tqdm",
    "google.generativeai": "google-generativeai",
    "PIL": "Pillow",
}
def module_is_available(module_name):
    try:
        return importlib.util.find_spec(module_name) is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        return False

missing = [package for module, package in packages.items() if not module_is_available(module)]
if missing:
    print("Installing missing Python packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

# GitHub's workflow installs FFmpeg and fonts using apt. Colab keeps its familiar setup.
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    if IS_GITHUB_ACTIONS:
        raise RuntimeError("FFmpeg/ffprobe are missing. Install them in the Part 3 GitHub Actions workflow.")
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg", "fonts-dejavu-core"])

FONT_PATH = Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf")
if not FONT_PATH.is_file():
    if IS_GITHUB_ACTIONS:
        raise RuntimeError("DejaVu font is missing. Install fonts-dejavu-core in the Part 3 workflow.")
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "fonts-dejavu-core"])

import gdown

# INPUT BEHAVIOR:
# - GitHub Actions: clips/ is restored from the successful Part 2 artifact. Do not fetch it again.
# - Google Colab: preserve the original Google Drive assets + clips folder downloads.
if IS_GITHUB_ACTIONS:
    assets_url = os.environ.get("DRIVE_ASSETS_URL", "").strip()
    if not assets_url:
        raise RuntimeError("Missing GitHub Actions secret DRIVE_ASSETS_URL (public Google Drive assets folder URL).")
    print("Downloading Part 3 assets from Google Drive into:", ASSETS_DIR)
    assets_download = gdown.download_folder(
        url=assets_url,
        output=str(ASSETS_DIR),
        quiet=False,
        use_cookies=False,
    )
    if not assets_download and not any(p.is_file() for p in ASSETS_DIR.rglob("*")):
        raise RuntimeError("Assets download failed. Check DRIVE_ASSETS_URL and set the folder sharing to Anyone with the link — Viewer.")
    print("Part 2 clips will be read from the downloaded GitHub artifact; no second clip download is performed.")
else:
    ASSETS_FOLDER_ID = "1l9ayaXiRBwUr1Hrjag7r3CycVDiN24fh"
    CLIPS_FOLDER_ID = "10DSjb9etdFzQTg_rftRJpJkC5jKBjsNT"
    print("Downloading original Colab assets folder...")
    gdown.download_folder(id=ASSETS_FOLDER_ID, output=str(ASSETS_DIR), quiet=False, use_cookies=False)
    print("Downloading original Colab clips folder...")
    gdown.download_folder(id=CLIPS_FOLDER_ID, output=str(INPUT_DIR), quiet=False, use_cookies=False)

print("\\nPart 3 setup complete.")
print("Workspace:", WORKSPACE)
print("Input folder:", INPUT_DIR)
print("Assets folder:", ASSETS_DIR)
print("Output folder:", OUTPUT_DIR)
print("FFmpeg:", shutil.which("ffmpeg"))
print("FFprobe:", shutil.which("ffprobe"))

#!/usr/bin/env python
# coding: utf-8

import subprocess, os, glob, time, json, re, random, shutil, sys
from pathlib import Path
from tqdm import tqdm
try:
    import google.generativeai as genai
except ImportError:
    genai = None
from PIL import Image, ImageDraw, ImageOps
from concurrent.futures import ThreadPoolExecutor

# ================= 1. SETUP & API KEY =================
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    print("⚠️ Warning: GEMINI_API_KEY not found in environment secrets. Using fallback default texts.")

try:
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel("gemini-2.5-flash")
except Exception:
    model = None

# ================= 2. PATHS & SETTINGS =================
# GitHub Actions downloads the Part 2 artifact into <workspace>/clips.
# Colab uses /content (or its current working directory) and can download clips itself.
BASE_PATH = str(WORKSPACE)
INPUT_DIR = str(WORKSPACE / "clips")
OUTPUT_DIR = str(WORKSPACE / "Ready_!")
ASSETS_DIR = str(WORKSPACE / "assets")
FONT = os.environ.get(
    "PART3_FONT_PATH",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
)
PROCESSED_LOGO = str(WORKSPACE / ".part3_logo_processed.png")

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
all_video_paths = sorted(
    glob.glob(os.path.join(INPUT_DIR, "**", "*.mp4"), recursive=True)
)
print(f"Part 3 workspace: {BASE_PATH}")
print(f"Part 3 input:     {INPUT_DIR}")
print(f"Part 3 assets:    {ASSETS_DIR}")
print(f"Part 3 output:    {OUTPUT_DIR}")
print(f"Part 2 MP4 files detected: {len(all_video_paths)}")

# --- ADJUST THESE VARIABLES TO CHANGE VIDEO DIMENSIONS ---
OUT_W, OUT_H = 1080, 1920
WEBCAM_W, WEBCAM_H = 9800, 800

# Change these two variables to adjust the center video height or width
# GAME_W is set to 1064 so that with +16 padding, the total width is exactly 1080 (100% width) and borders are clearly visible
GAME_W = 1034
GAME_H = 1040

def clean_for_ffmpeg(text):
    return re.sub(r'[^a-zA-Z0-9 ]', '', text)

def get_ai_text(video_path, vid_id):
    if not model or not GEMINI_API_KEY:
        return "AMAZING MOMENT", "WATCH UNTIL END"
    frame_path = f"temp_frame_{vid_id}.jpg"
    subprocess.run(["ffmpeg", "-y", "-i", video_path, "-ss", "00:00:02", "-vframes", "1", frame_path], capture_output=True)
    for attempt in range(2):
        try:
            time.sleep(2)
            img = Image.open(frame_path)
            prompt = "Analyze this image for a viral short. Give me 1 catchy top line (max 3 words) and 1 emotional bottom line (max 5 words). Return JSON: {'top': '...', 'sub': '...'}"
            response = model.generate_content([prompt, img])
            data = json.loads(response.text.replace("```json", "").replace("```", "").strip())
            return clean_for_ffmpeg(data['top']).upper(), clean_for_ffmpeg(data['sub']).upper()
        except:
            continue
    return "AMAZING MOMENT", "WATCH UNTIL END"

def prepare_circular_logo(logo_path):
    if not logo_path or not os.path.exists(logo_path):
        return False
    try:
        img = Image.open(logo_path).convert("RGBA").resize((150, 150), Image.Resampling.LANCZOS)
        mask = Image.new("L", (150, 150), 0)
        draw = ImageDraw.Draw(mask)
        draw.ellipse((4, 4, 146, 146), fill=255)

        output = Image.new("RGBA", (150, 150), (0, 0, 0, 0))
        output.paste(img, (0, 0), mask=mask)

        draw_border = ImageDraw.Draw(output)
        draw_border.ellipse((4, 4, 146, 146), outline="yellow", width=4)

        output.save(PROCESSED_LOGO, "PNG")
        return True
    except Exception as e:
        print(f"⚠️ Error processing logo: {e}")
        return False

def process_single_video(INPUT):
    category = os.path.basename(os.path.dirname(INPUT))
    rel_path = os.path.relpath(INPUT, INPUT_DIR)
    FINAL_OUT = os.path.join(OUTPUT_DIR, rel_path).replace(" ", "_")
    os.makedirs(os.path.dirname(FINAL_OUT), exist_ok=True)

    vid_name = os.path.basename(INPUT)
    vid_id = abs(hash(vid_name))

    print(f"\n✨ Pro-Processing [{category.upper()}]: {vid_name}")
    AI_TOP, AI_SUB = get_ai_text(INPUT, vid_id)

    # Resolve paths matching the category
    webcam_candidates = glob.glob(os.path.join(BASE_PATH, "assets", "web_cam", category, "*.mp4")) \
                        + glob.glob(os.path.join(BASE_PATH, "assets", "intro", category, "*.mp4")) \
                        + glob.glob(os.path.join(BASE_PATH, "assets", "top_clips", category, "*.mp4"))
    resolved_webcam = random.choice(webcam_candidates) if webcam_candidates else None

    bg_candidates = glob.glob(os.path.join(BASE_PATH, "assets", "Bg_Pic", "*.*"))
    bg_candidates = [f for f in bg_candidates if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    resolved_bg = random.choice(bg_candidates) if bg_candidates else None

    logo_candidates = glob.glob(os.path.join(BASE_PATH, "assets", "logos", category, "*.png"))
    resolved_logo_path = random.choice(logo_candidates) if logo_candidates else None

    # Resolve category-specific outro
    outro_candidates = glob.glob(os.path.join(BASE_PATH, "assets", "outro", category, "*.mp4"))
    resolved_outro = random.choice(outro_candidates) if outro_candidates else None

    temp_main_raw = f"temp_main_raw_{vid_id}.mp4"
    temp_main = f"temp_main_{vid_id}.mp4"
    temp_black = f"temp_black_{vid_id}.mp4"
    temp_outro = f"temp_outro_{vid_id}.mp4"

    try:
        dur_cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", INPUT]
        dur = float(subprocess.check_output(dur_cmd).decode().strip())
        new_dur = dur

        audio_check = subprocess.run(["ffprobe", "-show_streams", "-select_streams", "a", "-loglevel", "error", INPUT], capture_output=True, text=True)
        has_audio = len(audio_check.stdout) > 0
        audio_input = "[0:a]" if has_audio else "anullsrc=channel_layout=stereo:sample_rate=44100[a_dummy];[a_dummy]"

        ffmpeg_inputs = ["-i", INPUT]

        has_bg_pic = resolved_bg is not None and os.path.exists(resolved_bg)
        if has_bg_pic:
            ffmpeg_inputs.extend(["-loop", "1", "-i", resolved_bg])
            bg_input_index = 1
        else:
            bg_input_index = None

        has_webcam = resolved_webcam is not None and os.path.exists(resolved_webcam)
        if has_webcam:
            webcam_input_index = (bg_input_index if bg_input_index is not None else 0) + 1
            ffmpeg_inputs.extend(["-i", resolved_webcam])
        else:
            webcam_input_index = None

        has_logo = prepare_circular_logo(resolved_logo_path)
        if has_logo:
            logo_input_index = (webcam_input_index if webcam_input_index is not None else (bg_input_index if bg_input_index is not None else 0)) + 1
            ffmpeg_inputs.extend(["-loop", "1", "-i", PROCESSED_LOGO])
        else:
            logo_input_index = None

        # Background layer setup with 40% blur (boxblur=40)
        if has_bg_pic:
            bg_setup = f"[{bg_input_index}:v]scale={OUT_W}:{OUT_H}:force_original_aspect_ratio=increase,crop={OUT_W}:{OUT_H},boxblur=40[bg_layer];"
        else:
            bg_setup = f"[0:v]scale={OUT_W}:{OUT_H}:force_original_aspect_ratio=increase,crop={OUT_W}:{OUT_H},boxblur=40[bg_layer];"

        # Assemble layout overlays
        if has_webcam:
            filter_complex = (
                f"{bg_setup}"
                f"[0:v]scale={GAME_W}:{GAME_H}:force_original_aspect_ratio=increase,crop={GAME_W}:{GAME_H},pad={GAME_W}+16:{GAME_H}+16:8:8:color=0xFFD700[fg_bordered];"
                f"[{webcam_input_index}:v]scale={WEBCAM_W}:{WEBCAM_H}:force_original_aspect_ratio=increase,crop={WEBCAM_W}:{WEBCAM_H},pad={WEBCAM_W}+16:{WEBCAM_H}+16:8:8:color=0x8A2BE2[webcam_bordered];"
                f"[bg_layer][webcam_bordered]overlay=x=(W-w)/2:y=120:shortest=1[bg_with_webcam];"
                f"[bg_with_webcam][fg_bordered]overlay=x=(W-w)/2:y=970:shortest=1[base];"
            )
        else:
            filter_complex = (
                f"{bg_setup}"
                f"[0:v]scale={GAME_W}:{GAME_H}:force_original_aspect_ratio=increase,crop={GAME_W}:{GAME_H},pad={GAME_W}+16:{GAME_H}+16:8:8:color=0xFFD700[fg_bordered];"
                f"[bg_layer][fg_bordered]overlay=x=(W-w)/2:y=(H-h)/2:shortest=1[base];"
            )

        if has_logo:
            overlay_x_expr = "W-150-((W-150)*mod(t,5)/5)"
            overlay_y_expr = "(H-150)*mod(t,5)/5"
            filter_complex += f"[base][{logo_input_index}:v]overlay=x='{overlay_x_expr}':y='{overlay_y_expr}':shortest=1[post_logo];[post_logo]"
        else:
            filter_complex += "[base]"

        filter_complex += (
            f"drawtext=fontfile='{FONT}':text='LIKE • SHARE • COMMENT':fontsize=40:fontcolor=white:borderw=3:bordercolor=0x00FFFF:shadowx=2:shadowy=2:x=(w-text_w)/2:y=h-190,"
            f"drawtext=fontfile='{FONT}':text='FOLLOW FOR MORE':fontsize=45:fontcolor=white:borderw=3:bordercolor=red:shadowx=2:shadowy=2:x=(w-text_w)/2:y=h-130,"
            f"drawtext=fontfile='{FONT}':text='RAJPUT ADS AGENCY':fontsize=65:fontcolor=yellow:borderw=5:bordercolor=0xFFD700:shadowx=3:shadowy=3:x=(w-text_w)/2:y=h-65,format=yuv420p[v_final];"
            f"{audio_input}aresample=44100,pan=stereo|c0=c0|c1=c1,aformat=sample_fmts=fltp:sample_rates=44100:channel_layouts=stereo[a_orig]"
        )

        # Generate main raw formatted clip
        subprocess.run(["ffmpeg", "-y"] + ffmpeg_inputs + [
            "-filter_complex", filter_complex,
            "-map", "[v_final]", "-map", "[a_orig]",
            "-t", str(new_dur), "-r", "25", "-video_track_timescale", "25000",
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-preset", "ultrafast", "-crf", "23", "-c:a", "aac", "-b:a", "128k", "-ar", "44100", "-ac", "2", "-movflags", "+faststart", temp_main_raw
        ], check=True)

        # Standardize main clip to clean stream mappings for concatenation
        subprocess.run([
            "ffmpeg", "-y", "-i", temp_main_raw,
            "-filter_complex", "[0:v]scale=1080:1920,format=yuv420p,setsar=1[v];[0:a]aresample=44100,pan=stereo|c0=c0|c1=c1,aformat=sample_fmts=fltp:sample_rates=44100:channel_layouts=stereo[a]",
            "-map", "[v]", "-map", "[a]", "-r", "25", "-video_track_timescale", "25000",
            "-c:v", "libx264", "-preset", "ultrafast", "-crf", "23", "-c:a", "aac", "-b:a", "128k", "-ar", "44100", "-ac", "2", temp_main
        ], check=True)

        # Generate 1-second black screen gap with exact matching properties
        subprocess.run([
            "ffmpeg", "-y", "-f", "lavfi", "-i", f"color=c=black:s={OUT_W}x{OUT_H}:r=25:d=1",
            "-f", "lavfi", "-i", "anullsrc=cl=stereo:r=44100",
            "-t", "1", "-video_track_timescale", "25000",
            "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "ultrafast", "-crf", "23",
            "-c:a", "aac", "-b:a", "128k", "-ar", "44100", "-ac", "2", temp_black
        ], check=True)

        has_outro_video = resolved_outro is not None and os.path.exists(resolved_outro)
        if has_outro_video:
            # Check if the outro video file actually contains any audio stream
            outro_audio_check = subprocess.run(["ffprobe", "-show_streams", "-select_streams", "a", "-loglevel", "error", resolved_outro], capture_output=True, text=True)
            outro_has_audio = len(outro_audio_check.stdout) > 0

            if outro_has_audio:
                outro_filter = f"[0:v]scale={OUT_W}:{OUT_H}:force_original_aspect_ratio=decrease,pad={OUT_W}:{OUT_H}:(ow-iw)/2:(oh-ih)/2,format=yuv420p,setsar=1[v];[0:a]aresample=44100,pan=stereo|c0=c0|c1=c1,aformat=sample_fmts=fltp:sample_rates=44100:channel_layouts=stereo[a]"
                outro_inputs = ["-i", resolved_outro]
            else:
                outro_filter = f"[0:v]scale={OUT_W}:{OUT_H}:force_original_aspect_ratio=decrease,pad={OUT_W}:{OUT_H}:(ow-iw)/2:(oh-ih)/2,format=yuv420p,setsar=1[v];[1:a]aresample=44100,pan=stereo|c0=c0|c1=c1,aformat=sample_fmts=fltp:sample_rates=44100:channel_layouts=stereo[a]"
                outro_inputs = ["-i", resolved_outro, "-f", "lavfi", "-i", "anullsrc=cl=stereo:r=44100"]

            # Standardize outro format perfectly matching the dimensions, pixel formats, and audio layouts
            subprocess.run(["ffmpeg", "-y"] + outro_inputs + [
                "-filter_complex", outro_filter,
                "-map", "[v]", "-map", "[a]",
                "-t", "10", "-r", "25", "-video_track_timescale", "25000",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "ultrafast", "-crf", "23",
                "-c:a", "aac", "-b:a", "128k", "-ar", "44100", "-ac", "2", temp_outro
            ], check=True)

        # Concatenate intermediate clips securely
        if has_outro_video:
            concat_inputs = ["-i", temp_main, "-i", temp_black, "-i", temp_outro]
            concat_filter = "[0:v][0:a][1:v][1:a][2:v][2:a]concat=n=3:v=1:a=1[v][a]"
        else:
            concat_inputs = ["-i", temp_main, "-i", temp_black]
            concat_filter = "[0:v][0:a][1:v][1:a]concat=n=2:v=1:a=1[v][a]"

        subprocess.run(["ffmpeg", "-y"] + concat_inputs + [
            "-filter_complex", concat_filter,
            "-map", "[v]", "-map", "[a]",
            "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "ultrafast", "-crf", "23",
            "-c:a", "aac", "-b:a", "128k", "-ar", "44100", "-ac", "2", FINAL_OUT
        ], check=True)

        print(f"✅ Saved Successfully: {FINAL_OUT}")
    except Exception as e:
        print(f"❌ Save Error for {INPUT}: {e}")
        raise
    finally:
        # Clean up temporary files
        for tmp in [temp_main_raw, temp_main, temp_black, temp_outro]:
            if os.path.exists(tmp):
                try: os.remove(tmp)
                except: pass

if not all_video_paths:
    raise RuntimeError(
        f"NO VIDEOS FOUND under {INPUT_DIR}. The Part 2 artifact must be extracted "
        "into the workspace as clips/<CATEGORY>/*.mp4 before Part 3 starts."
    )

print(f"᎐᎐᎐ SAME-TO-SAME Editor Active | ᎐ Total: {len(all_video_paths)}")
with ThreadPoolExecutor(max_workers=1) as executor:
    list(tqdm(executor.map(process_single_video, all_video_paths), total=len(all_video_paths)))

output_videos = sorted(
    str(path) for path in Path(OUTPUT_DIR).rglob("*")
    if path.is_file() and path.suffix.lower() == ".mp4"
)
print(f"Part 3 output MP4 count: {len(output_videos)}")
if not output_videos:
    raise RuntimeError(
        f"Part 3 processed {len(all_video_paths)} input video(s) but created no MP4 files in {OUTPUT_DIR}."
    )
print("\n᎐᎐᎐ COMPLETED!")